# Predicting U.S. Financial Markets Using Oil Price Increase: The Role of Macroeconomic Conditions and Shock Types 

Can we improve the predictive power of oil price increases for U.S. financial markets by accounting for both the macroeconomic environment and the nature of the shocks driving these increases? 

# 1. Data Preparation

## 1.1 Imports and raw data loading

We load the two source files:
- `data_hec_project_1.xlsx`  daily and monthly macro-financial series
- `shock_types.xlsx` oil-shock classification (supply / demand / geopolitical)

Daily series will be resampled to **month-end** frequency so all variables share the same time index. The CFNAI (monthly) drives our observation frequency.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

daily_raw   = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Daily")
monthly_raw = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Monthly")
shocks_raw  = pd.read_excel("../data/external/shock_types.xlsx")

Daily columns : ['Dates', 'WTI', 'Brent  ', 'BCOM', 'TFT', 'NatGas', 'SP500', 'MSCIW', 'MSCIEM', 'R2000', 'US10', 'US2', 'HY', 'XAU']
Monthly columns: ['Dates', 'IP ', 'CFNAI', 'MISM', 'MISMPP', 'SISM', 'SIMSPP', 'USR', 'RFI']
Shocks columns : ['Start', 'End', 'Shock_type', 'Event']

Daily  : 9443 rows
Monthly: 435 rows
Shocks : 9 episodes


## 1.2 Inspect raw data

Before any transformation, we inspect each series for:
- data types (numeric vs. string-encoded NAs like `"#N/A N/A"`)
- date range coverage
- obvious gaps or anomalies

In [2]:
# ── Clean column names ─────────────────────────────────────────────────────────
daily_raw.columns   = daily_raw.columns.astype(str).str.strip()
monthly_raw.columns = monthly_raw.columns.astype(str).str.strip()
shocks_raw.columns  = shocks_raw.columns.astype(str).str.strip()

# ── Daily: set date index ─────────────────────────────────────────────────────
date_col = "Date" if "Date" in daily_raw.columns else "Dates"
daily_raw[date_col] = pd.to_datetime(daily_raw[date_col], errors="coerce")
daily = daily_raw.dropna(subset=[date_col]).set_index(date_col).sort_index()

# ── Inspect dtypes and NA patterns ───────────────────────────────────────────
print("=== Daily data: dtypes ===")
print(daily.dtypes, "\n")
print("=== Daily data: first 3 rows ===")
print(daily.head(3), "\n")
print("=== Daily data: sample of non-numeric values ===")
for c in daily.columns:
    non_num = daily[c][pd.to_numeric(daily[c], errors="coerce").isna() & daily[c].notna()]
    if len(non_num) > 0:
        print(f"  {c}: {non_num.unique()[:5]}  ({len(non_num)} occurrences)")

=== Daily data: dtypes ===
WTI       float64
Brent     float64
BCOM      float64
TFT       float64
NatGas    float64
SP500     float64
MSCIW     float64
MSCIEM    float64
R2000     float64
US10      float64
US2       float64
HY        float64
XAU       float64
dtype: object 

=== Daily data: first 3 rows ===
              WTI  Brent     BCOM  TFT  NatGas   SP500   MSCIW  MSCIEM  \
Dates                                                                    
1990-01-01  21.82  19.69  77.8405  NaN     NaN  353.40  567.34  214.69   
1990-01-02  22.89  19.90  81.0621  NaN     NaN  359.69  568.96  217.30   
1990-01-03  23.68  20.95  83.0504  NaN     NaN  358.76  569.10  220.45   

              R2000   US10    US2     HY     XAU  
Dates                                             
1990-01-01  168.228  7.935  7.841  16.08  401.25  
1990-01-02  169.943  7.930  7.875  16.08  399.00  
1990-01-03  170.780  7.974  7.927  16.08  395.00   

=== Daily data: sample of non-numeric values ===


In [3]:
# ── Monthly: set date index (normalise to end-of-month) ──────────────────────
md = monthly_raw.copy()
md["Dates"] = pd.to_datetime(md["Dates"], errors="coerce")
md = md.dropna(subset=["Dates"])
md["Dates"] = md["Dates"].dt.to_period("M").dt.to_timestamp("M")
md = md.set_index("Dates").sort_index()

print("=== Monthly data: dtypes ===")
print(md.dtypes, "\n")
print("=== Monthly data: first 3 rows ===")
print(md.head(3), "\n")
print(f"Coverage: {md.index.min()} → {md.index.max()}")

# Check for non-numeric values
print("\n=== Monthly: non-numeric entries ===")
for c in md.columns:
    non_num = md[c][pd.to_numeric(md[c], errors="coerce").isna() & md[c].notna()]
    if len(non_num) > 0:
        print(f"  {c}: {non_num.unique()[:5]}  ({len(non_num)} occurrences)")

=== Monthly data: dtypes ===
IP        float64
CFNAI     float64
MISM      float64
MISMPP    float64
SISM      float64
SIMSPP    float64
USR       float64
RFI       float64
dtype: object 

=== Monthly data: first 3 rows ===
                 IP  CFNAI  MISM  MISMPP  SISM  SIMSPP  USR  RFI
Dates                                                           
1989-12-31  62.0428  -0.01  47.4    47.4   NaN     NaN  NaN  NaN
1990-01-31  61.7290  -0.23  47.2    47.2   NaN     NaN  NaN  NaN
1990-02-28  62.2896   0.55  49.1    49.1   NaN     NaN  NaN  NaN 

Coverage: 1989-12-31 00:00:00 → 2026-02-28 00:00:00

=== Monthly: non-numeric entries ===


In [4]:
# ── Shock types: inspect ──────────────────────────────────────────────────────
print("=== Shock types ===")
print(shocks_raw.head(10))
print(f"\nShock type values: {sorted(shocks_raw['Shock_type'].dropna().unique())}")
print(f"  1 = Supply,  2 = Demand,  3 = Geopolitical")

=== Shock types ===
       Start        End  Shock_type                                      Event
0 1990-08-01 1991-02-01           3                                 "Gulf War"
1 1999-03-01 2000-09-01           2              "Global Recovery & OPEC cuts"
2 2003-01-01 2008-06-01           2        "Commodity Supercycle / China Boom"
3 2008-07-01 2009-03-01           2  "Global Financial Crisis Demand Collapse"
4 2011-02-01 2011-10-01           3           "Arab Spring / Libya disruption"
5 2014-09-01 2016-02-01           1       "Shale Supply Shock / OPEC strategy"
6 2020-03-01 2020-05-01           2                       "Covid Demand Shock"
7 2022-02-01 2022-10-01           3                 "Ukraine War Energy Shock"
8 2023-05-01 2024-03-01           1                    "OPEC+ Production Cuts"

Shock type values: [np.int64(1), np.int64(2), np.int64(3)]
  1 = Supply,  2 = Demand,  3 = Geopolitical


## 1.3 Clean and transform variables

**Transformation rules** (applied consistently):

| Variable type | Transformation | Rationale |
|---|---|---|
| **Price levels** (SP500, WTI, XAU, NatGas, MSCIEM) | Log-return: $r_t = \ln(P_t) - \ln(P_{t-1})$ | Makes returns stationary and comparable across scales |
| **Yields / rates** (US10, US2, HY) | First difference: $\Delta y_t = y_t - y_{t-1}$ | Yields are already in percentage form; log-returns are meaningless for rates |
| **Spreads** (CreditSpread = HY − US10, Slope = US10 − US2) | Level or first-difference | These are already stationary spreads |
| **Index levels** (IP, USR) | Log first-difference: $\Delta \ln(x_t)$ | Growth rate; makes trended series stationary |
| **Diffusion indices** (ISM, CFNAI) | Level (centred at 50 for ISM, 0 for CFNAI) | Already stationary by construction |

**Frequency alignment**: daily series → resample to last business day of each month → then transform. This ensures each monthly observation uses only information available at month-end (no look-ahead).

In [5]:
# ── Helper: clean string NAs and convert to numeric ──────────────────────────
def clean_numeric(series):
    """Replace Excel-style NA strings and coerce to float."""
    return series.replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")

# ═══════════════════════════════════════════════════════════════════════════════
# A. DAILY → MONTHLY price-based variables
# ═══════════════════════════════════════════════════════════════════════════════

# Clean daily price series
SP500_d  = clean_numeric(daily["SP500"])
WTI_d    = clean_numeric(daily["WTI"])
XAU_d    = clean_numeric(daily["XAU"])
NatGas_d = clean_numeric(daily["NatGas"])
MSCIEM_d = clean_numeric(daily["MSCIEM"])
HY_d     = clean_numeric(daily["HY"])
US10_d   = clean_numeric(daily["US10"])
US2_d    = clean_numeric(daily["US2"])

# Resample to end-of-month (last available observation per month)
# IMPORTANT: resample EACH series independently so a missing value in one
# series does not cause a gap in another (which would turn a 1-month diff
# into a multi-month diff).
SP500_m  = SP500_d.resample("ME").last()
WTI_m    = WTI_d.resample("ME").last()
XAU_m    = XAU_d.resample("ME").last()
NatGas_m = NatGas_d.resample("ME").last()
MSCIEM_m = MSCIEM_d.resample("ME").last()
HY_m     = HY_d.resample("ME").last()
US10_m   = US10_d.resample("ME").last()
US2_m    = US2_d.resample("ME").last()

# --- Log-returns for price series ---
Y       = np.log(SP500_m).diff().rename("Y")          # dependent variable
r_oil   = np.log(WTI_m).diff().rename("r_oil")
r_gold  = np.log(XAU_m).diff().rename("r_gold")
r_natgas = np.log(NatGas_m).diff().rename("r_natgas")
r_msciem = np.log(MSCIEM_m).diff().rename("r_msciem")

# --- First differences for yields ---
d_HY   = HY_m.diff().rename("d_HY")
d_US10 = US10_m.diff().rename("d_US10")
d_US2  = US2_m.diff().rename("d_US2")

# --- Spreads (level) ---
CreditSpread = (HY_m - US10_m).rename("CreditSpread")
Slope        = (US10_m - US2_m).rename("Slope")

print("Price-based monthly variables created.")
print(f"  Y (SP500 log-return)  : {Y.dropna().shape[0]} obs, {Y.dropna().index.min().date()} → {Y.dropna().index.max().date()}")
print(f"  r_oil (WTI log-return): {r_oil.dropna().shape[0]} obs")
print(f"  r_gold                : {r_gold.dropna().shape[0]} obs")
print(f"  r_msciem              : {r_msciem.dropna().shape[0]} obs")
print(f"  CreditSpread          : {CreditSpread.dropna().shape[0]} obs")
print(f"  Slope                 : {Slope.dropna().shape[0]} obs")

Price-based monthly variables created.
  Y (SP500 log-return)  : 434 obs, 1990-02-28 → 2026-03-31
  r_oil (WTI log-return): 434 obs
  r_gold                : 434 obs
  r_msciem              : 434 obs
  CreditSpread          : 435 obs
  Slope                 : 435 obs


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# B. MONTHLY macro variables (already at monthly frequency)
# ═══════════════════════════════════════════════════════════════════════════════

CFNAI = pd.to_numeric(md["CFNAI"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce").rename("CFNAI")
IP_m     = pd.to_numeric(md["IP"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
MISM_m   = pd.to_numeric(md["MISM"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
MISMPP_m = pd.to_numeric(md["MISMPP"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
USR_m    = pd.to_numeric(md["USR"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
RFI_m    = pd.to_numeric(md["RFI"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")

# Transformed monthly macro variables
d_log_IP  = np.log(IP_m).diff().rename("d_log_IP")     # IP growth
ISM_mfg   = (MISM_m - 50).rename("ISM_mfg")            # centred at 50 (>0 = expansion)
ISM_pp    = (MISMPP_m - 50).rename("ISM_pp")            # production plans
d_log_USR = np.log(USR_m).diff().rename("d_log_USR")    # US retail sales growth
RFI       = RFI_m.rename("RFI")

print("Monthly macro variables created.")
print(f"  CFNAI     : {CFNAI.dropna().shape[0]} obs")
print(f"  d_log_IP  : {d_log_IP.dropna().shape[0]} obs")
print(f"  ISM_mfg   : {ISM_mfg.dropna().shape[0]} obs")
print(f"  d_log_USR : {d_log_USR.dropna().shape[0]} obs")

Monthly macro variables created.
  CFNAI     : 435 obs
  d_log_IP  : 434 obs
  ISM_mfg   : 435 obs
  d_log_USR : 349 obs


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# C. MACRO-STATE DUMMIES (CFNAI-based)
# ═══════════════════════════════════════════════════════════════════════════════
# CFNAI > 0  → economy above trend → expansion
# CFNAI < -0.7 → economy well below trend → contraction (NBER-style threshold)
# Otherwise → neutral regime

D_exp = (CFNAI > 0).astype(int).rename("D_exp")
D_con = (CFNAI < -0.7).astype(int).rename("D_con")

print(f"D_exp = 1 for {D_exp.sum()} months  ({D_exp.mean():.1%})")
print(f"D_con = 1 for {D_con.sum()} months  ({D_con.mean():.1%})")

# ═══════════════════════════════════════════════════════════════════════════════
# D. OIL-SHOCK TYPE DUMMIES
# ═══════════════════════════════════════════════════════════════════════════════
shocks_raw["Start"] = pd.to_datetime(shocks_raw["Start"], errors="coerce")
shocks_raw["End"]   = pd.to_datetime(shocks_raw["End"], errors="coerce")
shocks_raw["Shock_type"] = pd.to_numeric(shocks_raw["Shock_type"], errors="coerce")
shocks = shocks_raw.dropna(subset=["Start", "End", "Shock_type"]).sort_values("Start")

# We need a monthly index to map shocks onto. Use Y's full index.
full_idx = Y.dropna().index
shock_type_series = pd.Series(0, index=full_idx, name="Shock_type")
month_start = full_idx.to_period("M").to_timestamp("M")

for _, row in shocks.iterrows():
    mask = (month_start >= row["Start"]) & (month_start <= row["End"])
    shock_type_series.loc[mask] = int(row["Shock_type"])

D_sup = (shock_type_series == 1).astype(int).rename("D_sup")
D_dem = (shock_type_series == 2).astype(int).rename("D_dem")
D_geo = (shock_type_series == 3).astype(int).rename("D_geo")

print(f"\nShock dummies (over {len(full_idx)} months with Y data):")
print(f"  D_sup (supply)      : {D_sup.sum()} months")
print(f"  D_dem (demand)      : {D_dem.sum()} months")
print(f"  D_geo (geopolitical): {D_geo.sum()} months")
print(f"  No shock            : {(shock_type_series == 0).sum()} months")

D_exp = 1 for 224 months  (51.5%)
D_con = 1 for 35 months  (8.0%)

Shock dummies (over 434 months with Y data):
  D_sup (supply)      : 27 months
  D_dem (demand)      : 93 months
  D_geo (geopolitical): 22 months
  No shock            : 292 months


## 1.4 Build the master monthly dataset

We now merge all variables into a single DataFrame. The **predictive lag** (`shift(1)` on regressors) will be applied at regression time, not here.  
This keeps the master dataset in contemporaneous form, making it reusable for any specification.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Master dataset (contemporaneous — lags applied at regression time)
# ═══════════════════════════════════════════════════════════════════════════════
master = pd.concat([
    # Dependent variable
    Y,
    # Oil
    r_oil,
    # Global market
    r_msciem, r_gold, r_natgas,
    # Financial conditions
    CreditSpread, Slope, d_HY, d_US10, d_US2,
    # Macro activity
    CFNAI, d_log_IP, ISM_mfg, ISM_pp, d_log_USR, RFI,
    # Regime dummies
    D_exp, D_con,
    D_sup, D_dem, D_geo,
], axis=1)

print(f"Master dataset: {master.shape[0]} rows × {master.shape[1]} columns")
print(f"Date range: {master.index.min().date()} → {master.index.max().date()}")
print(f"\nMissing values per variable:")
print(master.isna().sum().to_string())

Master dataset: 436 rows × 21 columns
Date range: 1989-12-31 → 2026-03-31

Missing values per variable:
Y                2
r_oil            2
r_msciem         2
r_gold           2
r_natgas         5
CreditSpread     1
Slope            1
d_HY             2
d_US10           2
d_US2            2
CFNAI            1
d_log_IP         2
ISM_mfg          1
ISM_pp           1
d_log_USR       87
RFI             48
D_exp            1
D_con            1
D_sup            2
D_dem            2
D_geo            2


C:\Users\danny\AppData\Local\Temp\ipykernel_22296\2675272221.py:4: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  master = pd.concat([


In [9]:
# ── Quick sanity check: correlations among key variables ──────────────────────
key_vars = ["Y", "r_oil", "r_msciem", "r_gold", "CreditSpread", "d_log_USR", "CFNAI"]
print("Pairwise correlations (contemporaneous):")
print(master[key_vars].corr().round(3).to_string())

Pairwise correlations (contemporaneous):
                  Y  r_oil  r_msciem  r_gold  CreditSpread  d_log_USR  CFNAI
Y             1.000  0.200     0.692   0.020        -0.216      0.222 -0.002
r_oil         0.200  1.000     0.241   0.136        -0.158      0.217  0.197
r_msciem      0.692  0.241     1.000   0.246        -0.128      0.113  0.016
r_gold        0.020  0.136     0.246   1.000        -0.001      0.018 -0.058
CreditSpread -0.216 -0.158    -0.128  -0.001         1.000     -0.039 -0.324
d_log_USR     0.222  0.217     0.113   0.018        -0.039      1.000 -0.350
CFNAI        -0.002  0.197     0.016  -0.058        -0.324     -0.350  1.000


## 1.5 Regression helper

We define a reusable function that:
1. Takes the master dataset, a list of regressor names, and lag specification.
2. Shifts regressors by 1 period (predictive: $Y_t = f(X_{t-1})$).
3. Drops rows with missing values.
4. Runs OLS with HAC (Newey–West) standard errors and reports key diagnostics.

This enforces a consistent predictive setup across all models and prevents look-ahead bias.

In [10]:
def predictive_ols(data, x_cols, y_col="Y", lag=1, hac=True, print_summary=True):
    """
    Run OLS:  Y_t = a + b * X_{t-lag} + e_t
    
    Parameters
    ----------
    data : DataFrame with contemporaneous y and x columns
    x_cols : list of regressor column names
    y_col : name of dependent variable (default "Y")
    lag : int, number of periods to lag regressors (default 1)
    hac : use Newey-West HAC standard errors (default True)
    
    Returns
    -------
    statsmodels OLS result object
    """
    df = data[[y_col] + x_cols].copy()
    # Lag regressors
    df_X = df[x_cols].shift(lag)
    df_pred = pd.concat([df[y_col], df_X], axis=1).dropna()
    
    y = df_pred[y_col]
    X = sm.add_constant(df_pred[x_cols])
    
    if hac:
        nobs = len(y)
        nw_lags = int(np.floor(4 * (nobs / 100) ** (2/9)))  # Newey-West bandwidth
        model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})
    else:
        model = sm.OLS(y, X).fit()
    
    if print_summary:
        print(model.summary())
        print(f"\nN = {model.nobs:.0f} | Adj. R² = {model.rsquared_adj:.4f} | AIC = {model.aic:.1f} | BIC = {model.bic:.1f}")
    
    return model


def vif_table(data, cols):
    """Compute Variance Inflation Factors for given columns."""
    X = data[cols].dropna()
    X = sm.add_constant(X)
    vifs = pd.Series(
        [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
        index=cols, name="VIF"
    )
    return vifs.round(2)


def residual_diagnostics(model, lags=5):
    """Print Ljung-Box and Breusch-Pagan test results."""
    resid = model.resid
    # Ljung-Box for serial correlation
    lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
    lb_p = lb["lb_pvalue"].values[0]
    # Breusch-Pagan for heteroskedasticity
    bp = het_breuschpagan(resid, model.model.exog)
    bp_p = bp[1]  # p-value of F-test
    
    print(f"Ljung-Box({lags}): stat={lb['lb_stat'].values[0]:.2f}, p={lb_p:.4f}  {'⚠ serial corr.' if lb_p < 0.05 else '✓ no serial corr.'}")
    print(f"Breusch-Pagan  : stat={bp[0]:.2f}, p={bp_p:.4f}  {'⚠ heterosked.' if bp_p < 0.05 else '✓ homoskedastic'}")
    
    return {"lb_pvalue": lb_p, "bp_pvalue": bp_p}

print("Helpers defined: predictive_ols(), vif_table(), residual_diagnostics()")

Helpers defined: predictive_ols(), vif_table(), residual_diagnostics()


# 2. Baseline Predictive Models (OLS)

## 2.1 Model A — Pure oil baseline

$$Y_t = \alpha + \beta \cdot r\_oil_{t-1} + \varepsilon_t$$

The simplest possible test: does last month's oil return predict this month's S&P 500 return?  
This is the benchmark against which all subsequent models will be compared.

In [11]:
# ── Model A: Y_t = a + b * r_oil_{t-1} ────────────────────────────────────────
model_A = predictive_ols(master, ["r_oil"])
print("\n── Residual diagnostics ──")
diag_A = residual_diagnostics(model_A)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.451
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.229
Time:                        14:46:19   Log-Likelihood:                 754.28
No. Observations:                 433   AIC:                            -1505.
Df Residuals:                     431   BIC:                            -1496.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0071      0.002      3.299      0.0

### Interpretation — Model A

**What was done**: Simple univariate predictive OLS. Only lagged oil return as regressor.

**Results**: 
- Lagged oil has near-zero predictive power for the S&P 500 in this unconditional specification. The Adj. R² is essentially zero.
- This does not mean oil is irrelevant — it means a blanket "oil went up last month" signal does not help forecast equities on average.

**Econometric reasoning**: 
- Efficient markets absorb most oil-price information within the same month. Any residual predictive power likely depends on *conditions* (macro regime, shock type) or on *controlling for confounders*.

**Next step**: Add global market controls to check whether oil's lack of significance is due to omitted variable bias.

## 2.2 Model B — Oil + global market controls

$$Y_t = \alpha + \beta_1 \cdot r\_oil_{t-1} + \beta_2 \cdot r\_msciem_{t-1} + \beta_3 \cdot r\_gold_{t-1} + \varepsilon_t$$

**Rationale**: Oil, emerging-market equities, and gold are all driven by global risk appetite and commodity demand. If oil's effect on U.S. equities comes from shared global exposure rather than an independent channel, adding `r_msciem` and `r_gold` will absorb it.

In [12]:
# ── Model B: Oil + global market controls ─────────────────────────────────────
model_B = predictive_ols(master, ["r_oil", "r_msciem", "r_gold"])
print("\n── Residual diagnostics ──")
diag_B = residual_diagnostics(model_B)
print("\n── VIF ──")
print(vif_table(master, ["r_oil", "r_msciem", "r_gold"]))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.123
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.339
Time:                        14:46:19   Log-Likelihood:                 755.01
No. Observations:                 433   AIC:                            -1502.
Df Residuals:                     429   BIC:                            -1486.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0073      0.002      3.311      0.0

### Interpretation — Model B

**What was done**: Added lagged MSCI EM and gold returns alongside oil.

**Key observations**:
- Oil, EM equities, and gold are all return series. At monthly frequency, these are essentially white noise — past returns should not predict future returns under market efficiency.
- Any significance here would suggest either a slow-diffusion channel or a risk-premium effect.

**Econometric reasoning**: 
- VIFs should be low (these are return series with modest correlation).
- If `r_msciem` absorbs all the R², it means global risk appetite — not oil specifically — contains whatever weak predictive signal exists.

**Next step**: Add financial condition variables (CreditSpread, Slope) to capture risk-premium channels that are more persistent and thus more likely to predict.

## 2.3 Model C — Oil + financial conditions

$$Y_t = \alpha + \beta_1 \cdot r\_oil_{t-1} + \beta_2 \cdot CreditSpread_{t-1} + \beta_3 \cdot Slope_{t-1} + \varepsilon_t$$

**Rationale**: CreditSpread (HY − US10) and Slope (US10 − US2) are well-documented predictors of equity returns in the literature (Fama & French, Campbell & Shiller). They capture risk premia and business-cycle expectations, and unlike return series, they are persistent — making them plausible leading indicators.

In [13]:
# ── Model C: Oil + financial conditions ────────────────────────────────────────
model_C = predictive_ols(master, ["r_oil", "CreditSpread", "Slope"])
print("\n── Residual diagnostics ──")
diag_C = residual_diagnostics(model_C)
print("\n── VIF ──")
print(vif_table(master, ["r_oil", "CreditSpread", "Slope"]))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.140
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.332
Time:                        14:46:19   Log-Likelihood:                 755.13
No. Observations:                 433   AIC:                            -1502.
Df Residuals:                     429   BIC:                            -1486.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0129      0.007      1.892   

### Interpretation — Model C

**What was done**: Replaced global-market returns with financial condition levels (CreditSpread, Slope).

**Key observations**:
- CreditSpread and Slope are *level* variables: they measure the current state of financial conditions, not shocks. A high credit spread signals elevated default risk and often precedes lower equity returns.
- These are more natural predictors than lagged returns, because they are persistent (slow-moving) and capture risk premia.

**Econometric reasoning**: 
- If CreditSpread is significant, it suggests a risk-premium channel that oil alone does not capture.
- Oil may still add information if it signals supply-side inflation risk beyond what credit markets price in.

**Next step**: Add macroeconomic activity variables to see if real-economy signals (retail sales, industrial production) contain leading information.

## 2.4 Model D — Oil + macro activity

$$Y_t = \alpha + \beta_1 \cdot r\_oil_{t-1} + \beta_2 \cdot d\_log\_USR_{t-1} + \beta_3 \cdot d\_log\_IP_{t-1} + \beta_4 \cdot ISM\_mfg_{t-1} + \varepsilon_t$$

**Rationale**: Real-economy variables move slower than financial prices and may carry genuine forward-looking information. In the earlier exploratory notebook, `d_log_USR` was the only consistently significant predictor. We test this systematically here.

In [14]:
# ── Model D: Oil + macro activity ──────────────────────────────────────────────
model_D = predictive_ols(master, ["r_oil", "d_log_USR", "d_log_IP", "ISM_mfg"])
print("\n── Residual diagnostics ──")
diag_D = residual_diagnostics(model_D)
print("\n── VIF ──")
print(vif_table(master, ["r_oil", "d_log_USR", "d_log_IP", "ISM_mfg"]))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     2.717
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0298
Time:                        14:46:19   Log-Likelihood:                 597.26
No. Observations:                 349   AIC:                            -1185.
Df Residuals:                     344   BIC:                            -1165.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0041      0.004      1.157      0.2

### Interpretation — Model D

**What was done**: Added real-economy predictors: retail sales growth, IP growth, and ISM manufacturing.

**Key observations**:
- `d_log_USR` is expected to be the strongest macro predictor: rising retail sales signals stronger consumer spending and a positive demand outlook.
- IP and ISM are coincident/lagging indicators — they may already be priced in by month-end.
- Oil may lose whatever marginal significance it had if macro fundamentals absorb its information.

**Econometric reasoning**: 
- If oil is significant alongside macro controls, it carries independent information (e.g., supply-side inflation channel).
- If only `d_log_USR` is significant, then what matters for forecasting U.S. equities is real-economy momentum, not oil per se.

**Next step**: Combine the best controls from models B–D into a single specification to identify the minimal set of useful predictors.

# 3. Control Selection — Combined Specification

## 3.1 Model E — Combined controls

We combine the most economically meaningful variables from each block:
- **Oil**: `r_oil` (the variable of interest)
- **Global market**: `r_msciem` (global equity proxy)
- **Financial conditions**: `CreditSpread` (credit risk premium)
- **Macro activity**: `d_log_USR` (unemployment growth — slow-moving real signal)

We intentionally omit: `r_gold` (correlated with `r_oil`), `Slope` (typically collinear with CreditSpread), `d_log_IP` / `ISM_mfg` (coincident indicators with high noise).

In [15]:
# ── Model E: Combined specification ───────────────────────────────────────────
combined_cols = ["r_oil", "r_msciem", "CreditSpread", "d_log_USR"]
model_E = predictive_ols(master, combined_cols)
print("\n── Residual diagnostics ──")
diag_E = residual_diagnostics(model_E)
print("\n── VIF ──")
print(vif_table(master, combined_cols))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     2.094
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0812
Time:                        14:46:19   Log-Likelihood:                 595.03
No. Observations:                 349   AIC:                            -1180.
Df Residuals:                     344   BIC:                            -1161.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0126      0.007      1.791   



N = 349 | Adj. R² = 0.0141 | AIC = -1180.1 | BIC = -1160.8

── Residual diagnostics ──
Ljung-Box(5): stat=3.61, p=0.6069  ✓ no serial corr.
Breusch-Pagan  : stat=80.02, p=0.0000  ⚠ heterosked.

── VIF ──
r_oil           1.20
r_msciem        1.16
CreditSpread    1.04
d_log_USR       1.05
Name: VIF, dtype: float64


### Interpretation — Model E

**What was done**: Combined best controls from each block into one parsimonious specification.

**Key observations**:
- This model tests whether oil retains any independent predictive power after conditioning on the key macro-financial variables.
- If only `d_log_USR` is significant, the implication is that real economy momentum — not oil — is the genuine leading indicator for U.S. equities.

**Econometric reasoning**: 
- Low VIFs confirm the variables capture distinct information dimensions.
- The question is not whether these variables *improve* R² (they may not), but whether oil *survives* as a predictor once we condition on them.

**Next step**: Before adding regime variables, test whether oil's predictive power depends on the macro environment using Frisch–Waugh–Lovell (FWL) decomposition.

## 3.2 Frisch–Waugh–Lovell test — Does oil carry independent information?

The FWL theorem allows us to isolate oil's *partial* contribution by:
1. Regressing $Y_t$ on controls$_{t-1}$ → residuals $\tilde{Y}_t$
2. Regressing $r\_oil_{t-1}$ on controls$_{t-1}$ → residuals $\tilde{r}\_oil_{t-1}$
3. Regressing $\tilde{Y}_t$ on $\tilde{r}\_oil_{t-1}$

The coefficient on $\tilde{r}\_oil$ is algebraically identical to the oil coefficient in the full model, but the FWL setup makes the interpretation transparent: it shows oil's effect *net of everything explained by controls*.

In [16]:
# ── FWL decomposition: isolate oil's partial predictive power ─────────────────
controls_fwl = ["r_msciem", "CreditSpread", "d_log_USR"]

# Predictive data (lag all regressors by 1)
df_fwl = master[["Y", "r_oil"] + controls_fwl].copy()
df_fwl_X = df_fwl[["r_oil"] + controls_fwl].shift(1)
df_fwl = pd.concat([df_fwl["Y"], df_fwl_X], axis=1).dropna()

Y_fwl     = df_fwl["Y"]
Z_fwl     = sm.add_constant(df_fwl[controls_fwl])
X_oil_fwl = df_fwl["r_oil"]

# Step 1-2: residualise Y and r_oil on controls
Y_resid    = sm.OLS(Y_fwl, Z_fwl).fit().resid
X_oil_resid = sm.OLS(X_oil_fwl, Z_fwl).fit().resid.rename("r_oil_resid")

# Step 3: FWL regression
fwl_result = sm.OLS(Y_resid, sm.add_constant(X_oil_resid)).fit()
print("── FWL: Y_resid ~ r_oil_resid (net of controls) ──")
print(fwl_result.summary())

# Verification: coefficient should match full model
print(f"\nFWL  coef on r_oil: {fwl_result.params['r_oil_resid']:.6f}")
print(f"Full coef on r_oil: {model_E.params['r_oil']:.6f}")
print(f"Match: {abs(fwl_result.params['r_oil_resid'] - model_E.params['r_oil']) < 1e-6}")

── FWL: Y_resid ~ r_oil_resid (net of controls) ──
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     2.697
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.101
Time:                        14:46:19   Log-Likelihood:                 595.03
No. Observations:                 349   AIC:                            -1186.
Df Residuals:                     347   BIC:                            -1178.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------

### Interpretation — FWL test

**What was done**: Decomposed oil's predictive contribution net of all other controls.

**Key observations**:
- The FWL coefficient on `r_oil_resid` is identical to the full-model coefficient (as guaranteed by the FWL theorem).
- The R² of the FWL regression shows how much variation in the *residualised* Y is explained by the *residualised* oil — i.e., oil's unique predictive contribution.
- If the FWL R² is near zero, oil adds essentially nothing beyond what macro-financial controls already provide.

**Decision for next step**: We now turn to regime-dependent specifications. Even if oil has no *unconditional* predictive power, it may predict equities conditional on the macroeconomic state or the type of oil shock.

# 4. Regime-Dependent Specifications

The research question asks whether accounting for the **macroeconomic environment** and **shock type** improves oil's predictive power. We test this through interaction terms.

## 4.1 Model F — Macro-regime interactions

$$Y_t = \alpha + \beta_1 \cdot r\_oil_{t-1} + \beta_2 \cdot (r\_oil \times D\_exp)_{t-1} + \beta_3 \cdot (r\_oil \times D\_con)_{t-1} + \gamma_1 D\_exp_{t-1} + \gamma_2 D\_con_{t-1} + \varepsilon_t$$

**Rationale**: Oil's effect on equities may differ in expansions vs. contractions. During expansions, oil increases may reflect strong demand (positive for stocks); during contractions, they signal supply shocks or cost pressures (negative for stocks).

In [17]:
# ── Model F: Macro-regime interactions ─────────────────────────────────────────
# Create interaction terms in master
master["r_oil_x_D_exp"] = master["r_oil"] * master["D_exp"]
master["r_oil_x_D_con"] = master["r_oil"] * master["D_con"]

regime_cols = ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]
model_F = predictive_ols(master, regime_cols)
print("\n── Residual diagnostics ──")
diag_F = residual_diagnostics(model_F)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     2.411
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0358
Time:                        14:46:19   Log-Likelihood:                 760.34
No. Observations:                 433   AIC:                            -1509.
Df Residuals:                     427   BIC:                            -1484.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0077      0.003      2.493

### Interpretation — Model F

**What was done**: Added expansion/contraction dummies and their interactions with oil.

**Key observations**:
- `r_oil` captures oil's effect in the *neutral* regime (D_exp = D_con = 0).
- `r_oil × D_exp` measures how much the oil-equity relationship *changes* during expansions.
- `r_oil × D_con` measures how much it changes during contractions.

**Interpretation of coefficients**:
- Oil effect in expansion: $\beta_1 + \beta_2$
- Oil effect in contraction: $\beta_1 + \beta_3$
- Oil effect in neutral: $\beta_1$

**Next step**: Add shock-type dummies to test whether the *origin* of the oil price change matters beyond the macro environment.

## 4.2 Model G — Shock-type interactions

$$Y_t = \alpha + \beta_1 r\_oil_{t-1} + \sum_k \delta_k (r\_oil \times D\_shock_k)_{t-1} + \sum_k \gamma_k D\_shock_{k,t-1} + \varepsilon_t$$

where $k \in \{sup, dem, geo\}$.

**Rationale**: Kilian (2009) argues that the equity-market response to oil depends on whether the oil price move is driven by supply disruptions, global demand, or geopolitical events. We test whether these distinctions matter for *prediction*.

In [18]:
# ── Model G: Shock-type interactions ───────────────────────────────────────────
master["r_oil_x_D_sup"] = master["r_oil"] * master["D_sup"]
master["r_oil_x_D_dem"] = master["r_oil"] * master["D_dem"]
master["r_oil_x_D_geo"] = master["r_oil"] * master["D_geo"]

shock_cols = ["r_oil", "D_sup", "D_dem", "D_geo",
              "r_oil_x_D_sup", "r_oil_x_D_dem", "r_oil_x_D_geo"]
model_G = predictive_ols(master, shock_cols)
print("\n── Residual diagnostics ──")
diag_G = residual_diagnostics(model_G)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     2.581
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0129
Time:                        14:46:19   Log-Likelihood:                 757.42
No. Observations:                 433   AIC:                            -1499.
Df Residuals:                     425   BIC:                            -1466.
Df Model:                           7                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0092      0.002      4.079

### Interpretation — Model G

**What was done**: Tested whether the type of oil shock (supply, demand, geopolitical) modifies oil's predictive relationship with equities.

**Key observations**:
- If shock-type interactions are significant, it implies that the oil-equity relationship is not homogeneous — the *reason* behind an oil price move matters.
- Geopolitical shocks (D_geo) are of particular interest because they are exogenous to the U.S. business cycle.

**Econometric reasoning**:
- With only a subset of months classified as shock episodes, interaction terms have limited variation. Small samples in each regime reduce statistical power.
- If many parameters are insignificant, this model is likely over-parameterised for a predictive setting.

**Next step**: Combine macro-regime and shock-type with the best controls from Model E for the fullest specification.

## 4.3 Model H — Full specification (regimes + controls)

This model merges the best controls (Model E) with macro-regime and shock-type interactions. It is the most comprehensive specification and tests the full research question.

**Warning**: this model has many parameters. We pay close attention to:
- parameter instability / sign flips
- multicollinearity (VIF)
- sample size after NA drops
- whether added complexity improves predictive power or just overfits

In [19]:
# ── Model H: Full specification ────────────────────────────────────────────────
full_cols = [
    # Oil
    "r_oil",
    # Macro-regime interactions
    "D_exp", "D_con", "r_oil_x_D_exp", "r_oil_x_D_con",
    # Shock-type (only D_geo — most exogenous and interesting)
    "D_geo", "r_oil_x_D_geo",
    # Controls
    "r_msciem", "CreditSpread", "d_log_USR",
]
model_H = predictive_ols(master, full_cols)
print("\n── Residual diagnostics ──")
diag_H = residual_diagnostics(model_H)
print("\n── VIF ──")
print(vif_table(master, full_cols))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     2.356
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0106
Time:                        14:46:19   Log-Likelihood:                 600.13
No. Observations:                 349   AIC:                            -1178.
Df Residuals:                     338   BIC:                            -1136.
Df Model:                          10                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0152      0.007      2.049



N = 349 | Adj. R² = 0.0255 | AIC = -1178.3 | BIC = -1135.9

── Residual diagnostics ──
Ljung-Box(5): stat=5.20, p=0.3918  ✓ no serial corr.
Breusch-Pagan  : stat=83.58, p=0.0000  ⚠ heterosked.

── VIF ──
r_oil            3.49
D_exp            1.14
D_con            1.56
r_oil_x_D_exp    2.38
r_oil_x_D_con    2.10
D_geo            1.06
r_oil_x_D_geo    1.10
r_msciem         1.21
CreditSpread     1.61
d_log_USR        1.12
Name: VIF, dtype: float64


### Interpretation — Model H

**What was done**: Fullest OLS specification with oil, macro-regime interactions, geopolitical shock interaction, and the core controls.

**Key observations**:
- This tests the complete research hypothesis: oil's predictive power may be conditional on both the macro environment *and* the shock type.
- We deliberately kept only D_geo from shock types (dropping D_sup and D_dem) to maintain parsimony. Supply and demand shocks overlap with macro conditions and showed limited contribution in Model G.

**Econometric reasoning**:
- If regime interactions are significant here but were not in Model F (without controls), it means the interactions become more detectable once we condition on the right confounders.
- If regime interactions are insignificant even in this rich specification, the conclusion is clear: oil does not have exploitable predictive power at the 1-month horizon, regardless of conditions.

**Next step**: Move to dynamic models (AR, VAR) to check whether accounting for autoregressive structure changes the conclusion.

# 5. Dynamic Models

## 5.1 AR lag selection

Before adding oil and controls, we check whether S&P 500 monthly returns exhibit autocorrelation structure worth modelling. We fit AR(p) models for $p = 1, \ldots, 12$ and select by AIC/BIC.

If Y is approximately white noise (low autocorrelation), AR terms add no value and we can stay with static OLS. If Y has mild AR structure, we use ARX (AR + exogenous oil).

In [20]:
# ── AR lag selection ───────────────────────────────────────────────────────────
base = master[["Y", "r_oil"]].dropna()
p_max = 12
rows = []
for p in range(1, p_max + 1):
    d = base.copy()
    d["r_oil_lag1"] = d["r_oil"].shift(1)  # oil always at lag 1
    for lag in range(1, p + 1):
        d[f"Y_lag{lag}"] = d["Y"].shift(lag)
    d = d.dropna()
    X_ar = sm.add_constant(d[[f"Y_lag{lag}" for lag in range(1, p + 1)] + ["r_oil_lag1"]])
    m = sm.OLS(d["Y"], X_ar).fit()
    rows.append({"p": p, "AIC": round(m.aic, 2), "BIC": round(m.bic, 2), "Adj_R2": round(m.rsquared_adj, 4)})

ar_sel = pd.DataFrame(rows).set_index("p")
print(ar_sel.to_string())
best_aic = ar_sel["AIC"].idxmin()
best_bic = ar_sel["BIC"].idxmin()
print(f"\nBest p by AIC: {best_aic}")
print(f"Best p by BIC: {best_bic}")
print(f"\n→ We use p = {best_bic} (BIC prefers parsimony, suitable for prediction)")

        AIC      BIC  Adj_R2
p                           
1  -1502.74 -1490.53  0.0023
2  -1496.82 -1480.54  0.0009
3  -1492.76 -1472.43  0.0027
4  -1490.08 -1465.70  0.0007
5  -1484.15 -1455.72 -0.0007
6  -1480.35 -1447.88  0.0030
7  -1482.54 -1446.02  0.0063
8  -1479.39 -1438.84  0.0070
9  -1473.51 -1428.94  0.0058
10 -1469.24 -1420.65  0.0052
11 -1462.84 -1410.23  0.0026
12 -1458.03 -1401.40  0.0027

Best p by AIC: 1
Best p by BIC: 1

→ We use p = 1 (BIC prefers parsimony, suitable for prediction)


## 5.2 ARX(1) — Baseline with AR dynamics

$$Y_t = \alpha + \phi Y_{t-1} + \beta \cdot r\_oil_{t-1} + \varepsilon_t$$

This adds one lag of Y to the baseline. If equity returns have any momentum or mean-reversion at the monthly frequency, the AR term will capture it, and we can check whether oil's coefficient changes when controlling for Y's own dynamics.

In [21]:
# ── ARX(1): Y_t = a + phi*Y_{t-1} + b*r_oil_{t-1} ────────────────────────────
df_arx = master[["Y", "r_oil"]].copy()
df_arx["Y_lag1"]     = df_arx["Y"].shift(1)
df_arx["r_oil_lag1"] = df_arx["r_oil"].shift(1)
df_arx = df_arx[["Y", "Y_lag1", "r_oil_lag1"]].dropna()

X_arx = sm.add_constant(df_arx[["Y_lag1", "r_oil_lag1"]])
model_ARX1 = sm.OLS(df_arx["Y"], X_arx).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(model_ARX1.summary())
print(f"\nN = {model_ARX1.nobs:.0f} | Adj. R² = {model_ARX1.rsquared_adj:.4f}")

print("\n── Residual diagnostics ──")
_ = residual_diagnostics(model_ARX1)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.085
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.339
Time:                        14:46:19   Log-Likelihood:                 754.37
No. Observations:                 433   AIC:                            -1503.
Df Residuals:                     430   BIC:                            -1491.
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0069      0.002      3.023      0.0

## 5.3 ARX(1) + controls — Best OLS augmented with Y lag

$$Y_t = \alpha + \phi Y_{t-1} + \beta_1 r\_oil_{t-1} + \beta_2 r\_msciem_{t-1} + \beta_3 CreditSpread_{t-1} + \beta_4 d\_log\_USR_{t-1} + \varepsilon_t$$

This is the ARX version of Model E. If the AR term is insignificant, it confirms that monthly equity returns are essentially unpredictable from their own past — and the OLS framework is sufficient.

In [22]:
# ── ARX(1) + controls ─────────────────────────────────────────────────────────
arx_base = ["r_oil", "r_msciem", "CreditSpread", "d_log_USR"]
df_arx2 = master[["Y"] + arx_base].copy()
df_arx2["Y_lag1"] = df_arx2["Y"].shift(1)
# Lag all exogenous regressors
for c in arx_base:
    df_arx2[f"{c}_lag1"] = df_arx2[c].shift(1)
arx2_xcols = ["Y_lag1"] + [f"{c}_lag1" for c in arx_base]
df_arx2 = df_arx2[["Y"] + arx2_xcols].dropna()

X_arx2 = sm.add_constant(df_arx2[arx2_xcols])
model_ARX2 = sm.OLS(df_arx2["Y"], X_arx2).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(model_ARX2.summary())
print(f"\nN = {model_ARX2.nobs:.0f} | Adj. R² = {model_ARX2.rsquared_adj:.4f}")

print("\n── Residual diagnostics ──")
_ = residual_diagnostics(model_ARX2)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     1.730
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.127
Time:                        14:46:19   Log-Likelihood:                 595.15
No. Observations:                 349   AIC:                            -1178.
Df Residuals:                     343   BIC:                            -1155.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.0133      0.00



N = 349 | Adj. R² = 0.0119

── Residual diagnostics ──
Ljung-Box(5): stat=3.57, p=0.6125  ✓ no serial corr.
Breusch-Pagan  : stat=80.80, p=0.0000  ⚠ heterosked.


### Interpretation — AR / ARX models

**What was done**: Tested whether adding an AR(1) term for Y improves over static OLS.

**Key observations**:
- Monthly equity returns typically show very low autocorrelation (efficient market implication). If Y_lag1 is insignificant, the AR term is redundant.
- The oil coefficient should be comparable to Model E. If it changes substantially, it suggests confounding between oil predictability and Y's own momentum.

**Econometric reasoning**:
- If the Ljung-Box test on residuals from Model E (static OLS) shows no serial correlation, the AR term is not needed.
- Adding an insignificant AR term wastes a degree of freedom and hurts out-of-sample performance.

**Next step**: Test a bivariate VAR (Y, r_oil) to examine dynamic feedback between oil and equities.


## 5.4 VAR(p) — Oil–equity dynamic system

A VAR models **both** Y and r_oil as jointly determined:
$$\begin{pmatrix} Y_t \\ r\_oil_t \end{pmatrix} = A_1 \begin{pmatrix} Y_{t-1} \\ r\_oil_{t-1} \end{pmatrix} + \ldots + A_p \begin{pmatrix} Y_{t-p} \\ r\_oil_{t-p} \end{pmatrix} + \varepsilon_t$$

**What VAR adds over OLS**: It captures Granger causality in both directions — does oil predict Y, and does Y predict oil? It also accounts for indirect, multi-step dynamic effects.

**When VAR is useful**: When there is bidirectional feedback. If the oil→Y channel is the only one, a simple predictive OLS is more efficient.

In [23]:
# ── VAR: bivariate (Y, r_oil) ─────────────────────────────────────────────────
var_data = master[["Y", "r_oil"]].dropna()

var_model = VAR(var_data)
lag_sel = var_model.select_order(maxlags=12)
print(lag_sel.summary())

p_var = max(1, lag_sel.bic)  # BIC for parsimony; enforce min 1 lag (needed for Granger test)
print(f"\nSelected lag order (BIC): p = {p_var}")
var_result = var_model.fit(p_var)
print(var_result.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -10.96     -10.94*   1.743e-05     -10.95*
1       -10.97      -10.92   1.717e-05      -10.95
2      -10.98*      -10.88  1.706e-05*      -10.94
3       -10.97      -10.83   1.728e-05      -10.91
4       -10.98      -10.81   1.708e-05      -10.91
5       -10.97      -10.76   1.726e-05      -10.88
6       -10.96      -10.71   1.742e-05      -10.86
7       -10.95      -10.66   1.755e-05      -10.84
8       -10.94      -10.62   1.771e-05      -10.81
9       -10.93      -10.56   1.800e-05      -10.78
10      -10.91      -10.51   1.822e-05      -10.75
11      -10.91      -10.47   1.830e-05      -10.73
12      -10.91      -10.44   1.820e-05      -10.73
--------------------------------------------------

Selected lag order (BIC): p = 1
  Summary of Regression Results   
Model:                         VAR
Method:                       

In [24]:
# ── Granger causality tests ───────────────────────────────────────────────────
# Note: BIC often selects 0 lags for equity returns (near white noise).
# We enforce p >= 1 when fitting, so Granger tests are always valid.
print("── Granger causality: r_oil → Y ──")
gc_oil_to_y = var_result.test_causality("Y", causing="r_oil", kind="f")
print(gc_oil_to_y.summary())

print("\n── Granger causality: Y → r_oil ──")
gc_y_to_oil = var_result.test_causality("r_oil", causing="Y", kind="f")
print(gc_y_to_oil.summary())

# Impulse response: 12-month horizon
irf = var_result.irf(12)
print("\n── Impulse Response: oil shock → Y (cumulated over 12 months) ──")

for h in [1, 3, 6, 12]:    print(f"  h={h:2d}: {irf.irfs[h, 0, 1]:.5f}")

── Granger causality: r_oil → Y ──
Granger causality F-test. H_0: r_oil does not Granger-cause Y. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value    df   
----------------------------------------------
         3.012          3.852   0.083 (1, 860)
----------------------------------------------

── Granger causality: Y → r_oil ──
Granger causality F-test. H_0: Y does not Granger-cause r_oil. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value    df   
----------------------------------------------
         1.787          3.852   0.182 (1, 860)
----------------------------------------------

── Impulse Response: oil shock → Y (cumulated over 12 months) ──
  h= 1: -0.03480
  h= 3: -0.00060
  h= 6: -0.00000
  h=12: 0.00000


### Interpretation — VAR

**What was done**: Estimated a bivariate VAR(p) for Y and r_oil, tested Granger causality in both directions, and computed impulse responses.

**Key observations**:
- **Y equation**: Does lagged oil Granger-cause Y? If the p-value > 0.05, oil has no statistically significant dynamic predictive power — confirming the OLS baseline result.
- **r_oil equation**: Oil is typically autoregressive (its own lags predict it). Whether Y Granger-causes oil is secondary to our question but informative about feedback channels.
- **Impulse responses**: Even if coefficients are insignificant, the IRF shows the dynamic propagation of an oil shock into equities over time.

**What VAR adds**: It confirms whether the lack of oil predictability in single-equation OLS is consistent with a system-level view. If oil does not Granger-cause Y here either, the conclusion is robust.

**Econometric reasoning**:
- VAR is symmetric — it does not impose an economic structure about what causes what. Granger causality is a statistical concept (predictive precedence), not economic causality.
- The bivariate VAR deliberately excludes controls (CreditSpread, d_log_USR) to test the *raw* dynamic relationship, comparable to Model A.

**Next step**: Model comparison table and out-of-sample evaluation.

# 6. Model Comparison

## 6.1 In-sample comparison table

We summarise all OLS models on the same metrics: oil coefficient (sign, significance), Adj. R², AIC, BIC, sample size, residual diagnostics.

In [25]:
# ── In-sample comparison ──────────────────────────────────────────────────────
def model_row(name, model, oil_param="r_oil"):
    """Extract key metrics from an OLS result."""
    coef = model.params.get(oil_param, np.nan)
    pval = model.pvalues.get(oil_param, np.nan)
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    return {
        "Model": name,
        "β_oil": round(coef, 4),
        "p(oil)": round(pval, 4),
        "Sig": sig,
        "Adj.R²": round(model.rsquared_adj, 4),
        "AIC": round(model.aic, 1),
        "BIC": round(model.bic, 1),
        "N": int(model.nobs),
        "k": model.df_model,
    }

comparison = pd.DataFrame([
    model_row("A: Oil only",           model_A),
    model_row("B: Oil + global mkt",   model_B),
    model_row("C: Oil + fin. cond.",   model_C),
    model_row("D: Oil + macro",        model_D),
    model_row("E: Combined controls",  model_E),
    model_row("F: Macro regimes",      model_F),
    model_row("G: Shock types",        model_G),
    model_row("H: Full (regime+ctrl)", model_H),
    model_row("ARX(1) baseline",       model_ARX1, oil_param="r_oil_lag1"),
    model_row("ARX(1) + controls",     model_ARX2, oil_param="r_oil_lag1"),
])

print(comparison.to_string(index=False))

                Model   β_oil  p(oil) Sig  Adj.R²     AIC     BIC   N    k
          A: Oil only -0.0331  0.2284      0.0043 -1504.6 -1496.4 433  1.0
  B: Oil + global mkt -0.0332  0.1789      0.0030 -1502.0 -1485.7 433  3.0
  C: Oil + fin. cond. -0.0366  0.1404      0.0035 -1502.3 -1486.0 433  3.0
       D: Oil + macro -0.0475  0.1201      0.0266 -1184.5 -1165.3 349  4.0
 E: Combined controls -0.0395  0.1462      0.0141 -1180.1 -1160.8 349  4.0
     F: Macro regimes -0.1326  0.0015 ***  0.0227 -1508.7 -1484.3 433  5.0
       G: Shock types -0.0223  0.4894      0.0047 -1498.8 -1466.3 433  7.0
H: Full (regime+ctrl) -0.1133  0.0094 ***  0.0255 -1178.3 -1135.9 349 10.0
      ARX(1) baseline -0.0348  0.1694      0.0023 -1502.7 -1490.5 433  2.0
    ARX(1) + controls -0.0398  0.1442      0.0119 -1178.3 -1155.2 349  5.0


### Interpretation — Model comparison

The table above reveals the key story:

1. **Oil's coefficient** ($\beta_{oil}$): track how it changes across specifications. If it stays significant even after adding controls, oil has independent predictive power. If it loses significance, it was capturing shared variation with other variables.

2. **Adj. R² vs. N trade-off**: More complex models (F, G, H) have more parameters but may lose observations due to NA. If Adj. R² does not improve materially, the added complexity is not justified.

3. **BIC**: Penalises parameters more heavily than AIC. The model with the lowest BIC offers the best bias-variance trade-off for prediction.

4. **Sign stability**: If oil's sign flips across models, the estimate is unstable — likely driven by multicollinearity or small-sample artefacts.

## 6.2 Out-of-sample evaluation

In-sample fit can overstate predictive ability due to overfitting. We perform **expanding-window** (recursive) out-of-sample forecasting:

1. Start with the first $T_0$ observations as training set.
2. Estimate the model, forecast $Y_{T_0+1}$.
3. Expand the window by 1 month, re-estimate, forecast the next observation.
4. Repeat until the end of sample.
5. Compute **MSFE** (Mean Squared Forecast Error) and compare to a benchmark (historical mean).

The key metric is the **out-of-sample R²** (Campbell & Thompson, 2008):
$$R^2_{OOS} = 1 - \frac{\sum (Y_t - \hat{Y}_t)^2}{\sum (Y_t - \bar{Y}_t)^2}$$

where $\bar{Y}_t$ is the expanding-window historical mean (the "no-predictability" benchmark). $R^2_{OOS} > 0$ means the model beats the historical mean.

In [26]:
# ── Out-of-sample evaluation (expanding window) ──────────────────────────────
def oos_evaluation(data, x_cols, y_col="Y", min_train=120):
    """
    Expanding-window OOS forecasting.
    
    Parameters
    ----------
    data : DataFrame (contemporaneous)
    x_cols : list of regressor names
    min_train : minimum training window (months)
    
    Returns
    -------
    dict with MSFE, R²_OOS, and forecast series
    """
    df = data[[y_col] + x_cols].copy()
    # Create predictive dataset (lag X by 1)
    df_X = df[x_cols].shift(1)
    df_pred = pd.concat([df[y_col], df_X], axis=1).dropna()
    
    y_all = df_pred[y_col].values
    X_all = sm.add_constant(df_pred[x_cols]).values
    dates = df_pred.index
    T = len(y_all)
    
    if T <= min_train + 10:
        return None  # not enough observations
    
    forecasts = []
    actuals = []
    hist_means = []
    
    for t in range(min_train, T):
        y_train = y_all[:t]
        X_train = X_all[:t]
        
        # Model forecast
        try:
            model = sm.OLS(y_train, X_train).fit()
            y_hat = model.predict(X_all[t:t+1])[0]
        except Exception:
            continue
        
        # Historical mean benchmark
        y_bar = y_train.mean()
        
        forecasts.append(y_hat)
        actuals.append(y_all[t])
        hist_means.append(y_bar)
    
    forecasts = np.array(forecasts)
    actuals = np.array(actuals)
    hist_means = np.array(hist_means)
    
    msfe_model = np.mean((actuals - forecasts) ** 2)
    msfe_bench = np.mean((actuals - hist_means) ** 2)
    r2_oos = 1 - msfe_model / msfe_bench
    
    return {
        "MSFE": msfe_model,
        "MSFE_bench": msfe_bench,
        "R2_OOS": r2_oos,
        "n_forecasts": len(forecasts),
    }

# ── Evaluate key models ──────────────────────────────────────────────────────
oos_specs = {
    "A: Oil only":          ["r_oil"],
    "B: Oil + global mkt":  ["r_oil", "r_msciem", "r_gold"],
    "C: Oil + fin. cond.":  ["r_oil", "CreditSpread", "Slope"],
    "D: Oil + macro":       ["r_oil", "d_log_USR", "d_log_IP", "ISM_mfg"],
    "E: Combined controls": ["r_oil", "r_msciem", "CreditSpread", "d_log_USR"],
    "F: Macro regimes":     ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"],
    "H: Full (regime+ctrl)": ["r_oil", "D_exp", "D_con", "r_oil_x_D_exp", "r_oil_x_D_con",
                               "D_geo", "r_oil_x_D_geo", "r_msciem", "CreditSpread", "d_log_USR"],
}

oos_rows = []
for name, cols in oos_specs.items():
    result = oos_evaluation(master, cols, min_train=120)
    if result:
        oos_rows.append({"Model": name, **result})
    else:
        oos_rows.append({"Model": name, "MSFE": np.nan, "R2_OOS": np.nan, "n_forecasts": 0})

oos_df = pd.DataFrame(oos_rows)
print("Out-of-Sample Evaluation (expanding window, min_train=120 months):\n")
print(oos_df.to_string(index=False, float_format=lambda x: f"{x:.6f}" if abs(x) < 1 else f"{x:.4f}"))

Out-of-Sample Evaluation (expanding window, min_train=120 months):

                Model     MSFE  MSFE_bench    R2_OOS  n_forecasts
          A: Oil only 0.001969    0.001943 -0.013545          313
  B: Oil + global mkt 0.002010    0.001943 -0.034480          313
  C: Oil + fin. cond. 0.001995    0.001943 -0.026665          313
       D: Oil + macro 0.002323    0.002014 -0.153245          229
 E: Combined controls 0.002266    0.002014 -0.125115          229
     F: Macro regimes 0.002002    0.001943 -0.030207          313
H: Full (regime+ctrl) 0.002816    0.002014 -0.397929          229


### Interpretation — Out-of-sample results

**What was done**: Expanding-window out-of-sample forecasting for all key model specifications.

**R²_OOS interpretation**:
- $R^2_{OOS} > 0$: the model produces better forecasts than the simple historical mean.
- $R^2_{OOS} \approx 0$: the model adds no value — the historical mean is just as good.
- $R^2_{OOS} < 0$: the model **overfits** — it is worse than a naive benchmark.

**Key questions**:
1. Does ANY oil-based model beat the historical mean?
2. Do regime-dependent specifications improve OOS even if they look good in-sample?
3. Does adding controls help or hurt OOS (overfitting risk)?

In the equity forecasting literature, $R^2_{OOS}$ values of 0.5–2% are considered economically meaningful (Welch & Goyal, 2008; Campbell & Thompson, 2008).

---
# 7. Final Model — Answering the Research Question

> **Research question**: *Can we improve the predictive power of oil price increases for U.S. financial markets by accounting for both the macroeconomic environment and the nature of the shocks driving these increases?*

We answer this question by **comparing three nested specifications**, each adding one layer of conditioning:

| Specification | What it tests |
|---|---|
| **Baseline (Model A)** | $Y_t = \alpha + \beta \cdot r\_oil_{t-1} + \varepsilon_t$ — Does oil predict equities *unconditionally*? |
| **Macro-regime (Model F)** | $Y_t = \alpha + \beta_1 r\_oil_{t-1} + \beta_2 (r\_oil \times D\_exp)_{t-1} + \beta_3 (r\_oil \times D\_con)_{t-1} + \gamma D_{t-1} + \varepsilon_t$ — Does conditioning on the macro environment *improve* the oil signal? |
| **Final preferred (Model F*)** | Model F + `d_log_USR` — Does adding the strongest macro control alongside regime interactions produce the best specification? |

The logic is progressive: if Model F improves significantly over Model A, the answer to the research question is **yes** — the macroeconomic environment matters. We then test whether a Wald test rejects joint insignificance of the regime interactions.


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL MODEL COMPARISON — Three nested specifications
# ══════════════════════════════════════════════════════════════════════════════

# ── (1) Baseline: Model A — Oil only ─────────────────────────────────────────
print("=" * 72)
print("MODEL A  —  Baseline (unconditional oil)")
print("Y_t = a + b * r_oil_{t-1}")
print("=" * 72)
final_A = predictive_ols(master, ["r_oil"], print_summary=False)
print(final_A.summary())
print(f"\nN = {final_A.nobs:.0f} | Adj. R² = {final_A.rsquared_adj:.4f} | AIC = {final_A.aic:.1f} | BIC = {final_A.bic:.1f}")
_ = residual_diagnostics(final_A)

# ── (2) Macro-regime: Model F — Oil + regime interactions ────────────────────
print("\n" + "=" * 72)
print("MODEL F  —  Macro-regime interactions")
print("Y_t = a + b1*r_oil + b2*(r_oil x D_exp) + b3*(r_oil x D_con) + g1*D_exp + g2*D_con")
print("=" * 72)
final_F = predictive_ols(master, ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"],
                          print_summary=False)
print(final_F.summary())
print(f"\nN = {final_F.nobs:.0f} | Adj. R² = {final_F.rsquared_adj:.4f} | AIC = {final_F.aic:.1f} | BIC = {final_F.bic:.1f}")
_ = residual_diagnostics(final_F)

# ── (3) Preferred: Model F* — Regime + best macro control ────────────────────
print("\n" + "=" * 72)
print("MODEL F*  —  Final preferred specification")
print("Y_t = a + b1*r_oil + b2*(r_oil x D_exp) + b3*(r_oil x D_con)")
print("      + g1*D_exp + g2*D_con + b4*d_log_USR")
print("=" * 72)
final_Fstar = predictive_ols(master,
    ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con", "d_log_USR"],
    print_summary=False)
print(final_Fstar.summary())
print(f"\nN = {final_Fstar.nobs:.0f} | Adj. R² = {final_Fstar.rsquared_adj:.4f} | AIC = {final_Fstar.aic:.1f} | BIC = {final_Fstar.bic:.1f}")
_ = residual_diagnostics(final_Fstar)

# ══════════════════════════════════════════════════════════════════════════════
# WALD TEST: Are the regime interaction terms jointly significant?
# H0: beta(r_oil_x_D_exp) = beta(r_oil_x_D_con) = 0
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 72)
print("WALD TEST — Joint significance of macro-regime interactions")
print("H0: coef(r_oil x D_exp) = coef(r_oil x D_con) = 0")
print("=" * 72)

# Use use_f=True for F-statistic version
wald_Fstar = final_Fstar.wald_test("r_oil_x_D_exp = 0, r_oil_x_D_con = 0", use_f=True)
print(f"\nModel F*:")
print(wald_Fstar)
wald_Fstar_p = float(wald_Fstar.pvalue)
if wald_Fstar_p < 0.05:
    print("→ REJECT H0 at 5%: Regime interactions are jointly significant.")
    print("  Accounting for the macroeconomic environment DOES improve the oil-equity relationship.")
else:
    print("→ Cannot reject H0 at 5%: Regime interactions are not jointly significant.")

wald_F = final_F.wald_test("r_oil_x_D_exp = 0, r_oil_x_D_con = 0", use_f=True)
print(f"\nModel F:")
print(wald_F)
wald_F_p = float(wald_F.pvalue)
if wald_F_p < 0.05:
    print("→ REJECT H0 at 5%: Regime interactions are jointly significant even without macro controls.")


MODEL A  —  Baseline (unconditional oil)
Y_t = a + b * r_oil_{t-1}
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.451
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.229
Time:                        14:46:20   Log-Likelihood:                 754.28
No. Observations:                 433   AIC:                            -1505.
Df Residuals:                     431   BIC:                            -1496.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

In [48]:

# ══════════════════════════════════════════════════════════════════════════════
# PUBLICATION-READY COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════

def stars(p):
    if p < 0.01: return "***"
    elif p < 0.05: return "**"
    elif p < 0.1: return "*"
    return ""

def fmt_coef(model, var):
    """Format coefficient with stars, or blank if variable not in model."""
    if var not in model.params.index:
        return ""
    c = model.params[var]
    p = model.pvalues[var]
    return f"{c:.4f}{stars(p)}"

# Re-estimate A and F on the common restricted sample (N=349, same as F*)
# so that AIC/BIC are comparable across all three models
master_r = master.dropna(subset=["d_log_USR"])
final_A_r = predictive_ols(master_r, ["r_oil"], print_summary=False)
final_F_r = predictive_ols(master_r, ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"],
                           print_summary=False)

# All variables across the 3 models
all_vars = ["const", "r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con", "d_log_USR"]
labels = {
    "const": "Intercept",
    "r_oil": "r_oil (lagged)",
    "r_oil_x_D_exp": "r_oil × D_exp",
    "r_oil_x_D_con": "r_oil × D_con",
    "D_exp": "D_exp (expansion)",
    "D_con": "D_con (contraction)",
    "d_log_USR": "Δlog(USR) (lagged)",
}

rows = []
for var in all_vars:
    rows.append({
        "Variable": labels.get(var, var),
        "Model A": fmt_coef(final_A, var),
        "Model F": fmt_coef(final_F, var),
        "Model F*": fmt_coef(final_Fstar, var),
    })

# Add blank separator + diagnostics
rows.append({"Variable": "─" * 25, "Model A": "─" * 12, "Model F": "─" * 12, "Model F*": "─" * 12})

# Oil effect by regime (from Model F*)
b1 = final_Fstar.params["r_oil"]
b2 = final_Fstar.params["r_oil_x_D_exp"]
b3 = final_Fstar.params["r_oil_x_D_con"]
rows.append({"Variable": "Oil effect in NEUTRAL",     "Model A": f"{final_A.params['r_oil']:.4f}", "Model F": f"{final_F.params['r_oil']:.4f}", "Model F*": f"{b1:.4f}"})
rows.append({"Variable": "Oil effect in EXPANSION",   "Model A": "", "Model F": f"{final_F.params['r_oil'] + final_F.params['r_oil_x_D_exp']:.4f}", "Model F*": f"{b1+b2:.4f}"})
rows.append({"Variable": "Oil effect in CONTRACTION", "Model A": "", "Model F": f"{final_F.params['r_oil'] + final_F.params['r_oil_x_D_con']:.4f}", "Model F*": f"{b1+b3:.4f}"})

rows.append({"Variable": "─" * 25, "Model A": "─" * 12, "Model F": "─" * 12, "Model F*": "─" * 12})

# N for coefficients (full sample for A and F; restricted for F*)
rows.append({"Variable": "N (coefficients)",  "Model A": f"{final_A.nobs:.0f}", "Model F": f"{final_F.nobs:.0f}", "Model F*": f"{final_Fstar.nobs:.0f}"})
rows.append({"Variable": "Adj. R²",           "Model A": f"{final_A.rsquared_adj:.4f}", "Model F": f"{final_F.rsquared_adj:.4f}", "Model F*": f"{final_Fstar.rsquared_adj:.4f}"})

# AIC/BIC on the common restricted sample (N=349) for a valid comparison
rows.append({"Variable": "AIC  [common N=349]", "Model A": f"{final_A_r.aic:.1f}", "Model F": f"{final_F_r.aic:.1f}", "Model F*": f"{final_Fstar.aic:.1f}"})
rows.append({"Variable": "BIC  [common N=349]", "Model A": f"{final_A_r.bic:.1f}", "Model F": f"{final_F_r.bic:.1f}", "Model F*": f"{final_Fstar.bic:.1f}"})

# Wald p-value
rows.append({"Variable": "Wald (interactions=0)", "Model A": "—", "Model F": f"p={wald_F.pvalue:.4f}", "Model F*": f"p={wald_Fstar.pvalue:.4f}"})

# OOS R²
oos_A = oos_evaluation(master, ["r_oil"], min_train=120)
oos_F = oos_evaluation(master, ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"], min_train=120)
oos_Fstar = oos_evaluation(master, ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con", "d_log_USR"], min_train=120)
rows.append({"Variable": "R²_OOS", "Model A": f"{oos_A['R2_OOS']:.4f}" if oos_A else "—", 
             "Model F": f"{oos_F['R2_OOS']:.4f}" if oos_F else "—", 
             "Model F*": f"{oos_Fstar['R2_OOS']:.4f}" if oos_Fstar else "—"})

# Print
result_table = pd.DataFrame(rows)
print("\n" + "=" * 72)
print("        FINAL COMPARISON TABLE — Answering the Research Question")
print("=" * 72)
print("  Significance: *** p<0.01, ** p<0.05, * p<0.10")
print("  HAC (Newey-West) standard errors throughout")
print("  Predictive setup: Y_t = f(X_{t-1})")
print("  Note: Coefficients for A and F use full sample (N=433).")
print("  AIC/BIC re-estimated on the common restricted sample (N=349)")
print("  to ensure valid cross-model comparison.")
print("=" * 72)
print(result_table.to_string(index=False))
print("=" * 72)



        FINAL COMPARISON TABLE — Answering the Research Question
  Significance: *** p<0.01, ** p<0.05, * p<0.10
  HAC (Newey-West) standard errors throughout
  Predictive setup: Y_t = f(X_{t-1})
  Note: Coefficients for A and F use full sample (N=433).
  AIC/BIC re-estimated on the common restricted sample (N=349)
  to ensure valid cross-model comparison.
                 Variable      Model A      Model F     Model F*
                Intercept    0.0071***     0.0077**     0.0073**
           r_oil (lagged)      -0.0331   -0.1326***   -0.1107***
            r_oil × D_exp                 0.1615***     0.1264**
            r_oil × D_con                    0.1159       0.0837
        D_exp (expansion)                   -0.0000      -0.0014
      D_con (contraction)                   -0.0007      -0.0106
       Δlog(USR) (lagged)                               0.3422**
───────────────────────── ──────────── ──────────── ────────────
    Oil effect in NEUTRAL      -0.0331      -0.1326    

## 7.1 Reading the final comparison table

### What happens to oil across the three specifications?

| | Model A (baseline) | Model F (macro regime) | Model F* (regime + control) |
|---|---|---|---|
| **Oil coefficient** | −0.033 (p=0.228) | **−0.133 (p=0.002***)** | **−0.111 (p=0.009***)** |
| **Oil significant?** | No | **Yes** | **Yes** |
| **Adj. R²** | 0.004 | **0.023** | **0.022** |
| **Wald test (interactions=0)** | — | **p=0.011** | p=0.078 |

### The central result

**Oil is NOT significant unconditionally** (Model A, p=0.228). But once we condition on the macroeconomic regime, **oil becomes highly significant** (Model F, p=0.002\*\*\*). The Wald test firmly rejects the null that regime interactions are zero (p=0.011). This directly and positively answers the research question.

### Why does this happen? — The regime-dependent oil effect

The interaction terms reveal that oil's predictive power is **not uniform across economic conditions**:

| Macro regime | Oil effect (Model F) | Interpretation |
|---|---|---|
| **Neutral** ($D_{exp}=0, D_{con}=0$) | $\beta_1 = -0.133$*** | Oil increases predict equity *declines* — consistent with cost-push / supply-shock channel |
| **Expansion** ($D_{exp}=1$) | $\beta_1 + \beta_2 = -0.133 + 0.162 = +0.029$ | Oil effect **vanishes** — oil increases during expansions reflect strong demand, which is not bearish for stocks |
| **Contraction** ($D_{con}=1$) | $\beta_1 + \beta_3 = -0.133 + (n.s.) \approx -0.133$ | Oil effect stays **strongly negative** — oil increases during weak economies amplify the downturn signal |

**Economic intuition**: During expansions, rising oil prices are a symptom of strong global demand (positive for earnings). During contractions or neutral periods, rising oil prices signal cost pressures or supply disruptions (negative for earnings). The unconditional average across these opposite effects washes out to zero — which is why Model A finds nothing.

### Model F* adds `d_log_USR` (retail sales growth)

Model F* confirms that `d_log_USR` is independently significant (p=0.026\*\*), adding a real-economy channel beyond the regime-dependent oil effect. However, its inclusion weakens the Wald test for the interactions to p=0.078 (significant at 10% but not 5%). This is because retail sales growth partially captures the same business-cycle information as the regime dummies.

### Out-of-sample caveat

All three models have negative $R^2_{OOS}$ (−0.01, −0.03, −0.34). This is typical for monthly equity return prediction (Welch & Goyal, 2008) — in-sample statistical relationships do not reliably translate into out-of-sample forecasting gains. The finding is about **understanding the mechanism** (regime-dependent oil sensitivity), not about building a profitable trading model.

---

## 7.2 Final answer to the research question

> *Can we improve the predictive power of oil price increases for U.S. financial markets by accounting for both the macroeconomic environment and the nature of the shocks driving these increases?*

### **Yes — for the macroeconomic environment; No — for shock types.**

1. **Macroeconomic environment** (Model F): Accounting for the CFNAI-based expansion/contraction regime transforms oil from a non-predictor (p=0.228) into a **statistically significant** predictor (p=0.002\*\*\*) with a 5× increase in explanatory power (Adj. R²: 0.004 → 0.023). The Wald test confirms joint significance of the regime interactions (p=0.011). **This is the key finding of this analysis.**

2. **Shock types** (Model G from Section 4): Adding supply/demand/geopolitical dummies does NOT improve prediction (Adj. R² = 0.005, oil p=0.489). The shock classification adds too many parameters with too few observations per shock type at the monthly frequency.

3. **Best overall model**: **Model F** — the parsimonious macro-regime specification — is the preferred model. It delivers significant results, has the best BIC among conditional models, and directly answers the research question. Adding `d_log_USR` (Model F\*) provides additional insight but is not strictly necessary for answering the regime question.


---
# 8. Summary of the Full Progressive Analysis

This notebook progressively tested whether oil price changes can forecast U.S. equity returns, and whether this predictive power improves when conditioning on the macroeconomic environment and the type of oil-price shock.

### Sections 2–3: Baseline and control selection
- **Model A** (oil only): Oil is not a significant predictor unconditionally (p=0.228, Adj. R²≈0).
- **Models B–D**: Adding global market, financial conditions, or macro controls does not make oil significant. Only `d_log_USR` (retail sales growth) is independently significant.
- **Model E + FWL**: Oil's partial contribution (net of controls) has R²=0.008 — near zero.

### Section 4: Regime-dependent specifications
- **Model F** (macro regimes): Oil becomes **highly significant** (p=0.002\*\*\*) once we allow its effect to vary by expansion/contraction regime. Wald test rejects H0 at 5% (p=0.011).
- **Model G** (shock types): Supply/demand/geopolitical dummies do NOT improve prediction — too many parameters, too little variation per shock bucket at monthly frequency.
- **Model H** (full): Confirms oil significance persists with controls, but overfits out-of-sample.

### Section 5: Dynamic models
- AR/ARX: Monthly equity returns have negligible autocorrelation. AR terms add no value.
- VAR/Granger: Oil→Y Granger causality is marginal (p=0.083). The OLS framework is sufficient.

### Section 6: Out-of-sample
- No model beats the historical mean out-of-sample — typical for monthly equity prediction.

### Section 7: Final model
- **Model F is the preferred specification.** It directly answers the research question with statistically significant results, good parsimony, and a clear economic mechanism (regime-dependent oil sensitivity).


## FURTHER TESTS

In [ ]:

shocks_ext = pd.read_excel("../data/external/shock_extended.xlsx")
shocks_ext["Start"] = pd.to_datetime(shocks_ext["Start"], errors="coerce")
shocks_ext["End"]   = pd.to_datetime(shocks_ext["End"],   errors="coerce")

# Map shock episodes onto monthly index (same logic as original D_sup/D_dem/D_geo)
full_idx   = master.index
month_start = full_idx.to_period("M").to_timestamp("M")
shock_ext_series = pd.Series("none", index=full_idx)

for _, row in shocks_ext.dropna(subset=["Start", "End"]).iterrows():
    mask = (month_start >= row["Start"]) & (month_start <= row["End"])
    shock_ext_series.loc[mask] = row["Type_of_Shock"].strip().lower()

D_sup_ext = (shock_ext_series == "supply").astype(int)
D_dem_ext = (shock_ext_series == "demand").astype(int)
D_geo_ext = (shock_ext_series == "geopolitical").astype(int)

print(f"Extended shock dummies ({len(full_idx)} months):")
print(f"  Supply      : {D_sup_ext.sum()} months")
print(f"  Demand      : {D_dem_ext.sum()} months")
print(f"  Geopolitical: {D_geo_ext.sum()} months")

# Build interaction terms and run same spec as Model G
master["D_sup_ext"] = D_sup_ext
master["D_dem_ext"] = D_dem_ext
master["D_geo_ext"] = D_geo_ext
master["r_oil_x_D_sup_ext"] = master["r_oil"] * master["D_sup_ext"]
master["r_oil_x_D_dem_ext"] = master["r_oil"] * master["D_dem_ext"]
master["r_oil_x_D_geo_ext"] = master["r_oil"] * master["D_geo_ext"]

shock_ext_cols = ["r_oil", "D_sup_ext", "D_dem_ext", "D_geo_ext",
                  "r_oil_x_D_sup_ext", "r_oil_x_D_dem_ext", "r_oil_x_D_geo_ext"]

model_G_ext = predictive_ols(master, shock_ext_cols, hac=False)

# Wald test: shock-type interactions jointly zero?
wald_G_ext = model_G_ext.wald_test(["r_oil_x_D_sup_ext = 0",
                                     "r_oil_x_D_dem_ext = 0",
                                     "r_oil_x_D_geo_ext = 0"])
f_stat = float(np.asarray(wald_G_ext.statistic).item())
print(f"\nWald test (interactions = 0):  F = {f_stat:.3f},  p = {wald_G_ext.pvalue:.4f}")
print(f"Adj. R²  (Model G ext): {model_G_ext.rsquared_adj:.4f}")
print(f"Adj. R²  (Model G ori): {model_G.rsquared_adj:.4f}  (original 9-episode dataset)")


Extended shock dummies (436 months):
  Supply      : 53 months
  Demand      : 148 months
  Geopolitical: 50 months
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.472
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.175
Time:                        15:08:25   Log-Likelihood:                 758.04
No. Observations:                 433   AIC:                            -1500.
Df Residuals:                     425   BIC:                            -1468.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
--------

Pas de changement dans l interpretation, toujours pas significatif a part geo x oil

### Increases in oil shocks only

In [ ]:
master["r_oil_pos"] = master["r_oil"].clip(lower=0)
master["r_oil_neg"] = master["r_oil"].clip(upper=0)

master["r_oil_pos_x_D_exp"] = master["r_oil_pos"] * master["D_exp"]
master["r_oil_pos_x_D_con"] = master["r_oil_pos"] * master["D_con"]
master["r_oil_neg_x_D_exp"] = master["r_oil_neg"] * master["D_exp"]
master["r_oil_neg_x_D_con"] = master["r_oil_neg"] * master["D_con"]

model_asym = predictive_ols(master,
    ["r_oil_pos", "r_oil_neg",
     "r_oil_pos_x_D_exp", "r_oil_pos_x_D_con",
     "r_oil_neg_x_D_exp", "r_oil_neg_x_D_con",
     "D_exp", "D_con", "d_log_USR"],
    hac=False)


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.066
Model:                            OLS   Adj. R-squared:                  0.041
Method:                 Least Squares   F-statistic:                     2.669
Date:                Fri, 03 Apr 2026   Prob (F-statistic):            0.00528
Time:                        15:36:15   Log-Likelihood:                 602.49
No. Observations:                 349   AIC:                            -1185.
Df Residuals:                     339   BIC:                            -1146.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.0026      0.00

OIL DECREASES IMPACT NEGATIVELY IN NEUTRAL REGIME  
OIL DECREASES IN EXPANSION IMPACT POSITIVELY
Rest is not significant, hard to say in contractions  



## Analysis for different markets
SP500, Bonds, Credits.

### Bonds

First difference of the U.S. 10-year Treasury yield 

In [ ]:

print("R oil")
predictive_ols(master, ["r_oil", "D_exp", "D_con", "d_log_USR"], y_col="d_US10", hac=False)

print("\n\noil increases vs decreases")
predictive_ols(master, ["r_oil_pos", "r_oil_neg", "D_exp", "D_con", "d_log_USR"], y_col="d_US10", hac=False)


R oil
                            OLS Regression Results                            
Dep. Variable:                 d_US10   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.149
Date:                Fri, 03 Apr 2026   Prob (F-statistic):              0.333
Time:                        15:50:38   Log-Likelihood:                -20.751
No. Observations:                 349   AIC:                             51.50
Df Residuals:                     344   BIC:                             70.78
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0272      0.021     -1.297   

### Credit spread
First difference of the credit spread 

In [ ]:


master["d_CreditSpread"] = master["CreditSpread"].diff()

predictive_ols(master, ["r_oil", "D_exp", "D_con", "d_log_USR"], y_col="d_CreditSpread", hac=False)

print("\n\noil increases vs decreases")
predictive_ols(master, ["r_oil_pos", "r_oil_neg", "D_exp", "D_con", "d_log_USR"], y_col="d_CreditSpread", hac=False)


=== CREDIT: d_CreditSpread — symmetric oil ===
                            OLS Regression Results                            
Dep. Variable:         d_CreditSpread   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     2.566
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0381
Time:                        15:48:54   Log-Likelihood:                -378.16
No. Observations:                 349   AIC:                             766.3
Df Residuals:                     344   BIC:                             785.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const

Nothing interesting.